# TRAINING CODE OF DIAGNOSIS FOR MODELFLOWS-APP

## Setup

- Upload the following scripts, specified in the section 'Import local libraries':

  *   Utils/datasets.py
  *   Utils/utils.py
  *   Training/batch_generator.py
  *   Training/callbacks.py
  *   Training/model.py

- Upload the databases to be used for training and validation.

## Usage

- Only the section of 'input parameters' can be edited to configure the values of the hyperparameters.
- Once determined the values of the parameters, just run the rest of the code.

## Main

### Import libraries

In [1]:
from tensorflow import keras
import tensorflow as tf
import tensorflow_addons as tfa

import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 20})

import os

import numpy as np
import numpy.random as rng

np.random.seed(0)

2026-02-19 10:28:45.493689: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-19 10:28:45.694031: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-19 10:28:45.694877: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-19 10:28:46.631161: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/ander/miniconda3/envs/cardiac_modelFlows/lib/python3.10/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of 

### Import local libraries

In [2]:
from Training import batch_generator as batch_generator
from Training import callbacks as callbacks
from Training import model as model

from Utils import datasets as my_datasets

### Input parameters

In [3]:
param = {}

#### Data parameters

In [4]:
# PARAMETROS DE LAS RUTAS DEL CONJUNTO DE DATOS DE ENTRENAMIENTO Y VALIDACION
param['training_database_path'] = ['/home/ander/Escritorio/Workshop2025-2026/Databases/Mice_ecos/Ecos_tutorials_AI/ecos_orig/ecos_orig_Training']
param['validation_database_path'] = ['/home/ander/Escritorio/Workshop2025-2026/Databases/Mice_ecos/Ecos_tutorials_AI/ecos_orig/ecos_orig_Validation']

#### Model parameters

In [5]:
# PARAMETROS PARA EL TAMAÑO DE LA ENTRADA

# Cada imagen se redimensiona a 256 × 256 píxeles
param['target_size'] = (256, 256)

# Numero de canales, entonces la dimension real sera 256×256×1 (grises)
param['num_channels'] = 1

#---------------------------------------------------------------------------------------------------------------------------------------------

# PARAMETROS PARA EL ViT

# Tamaño del parche en el que se divide la imagen, la imagen de 256×256 se divide 
# en parches de 32×32. # de parches por dimension 256/32=8
param['patch_size'] = 32

# El Transformer tendrá 8 bloques encoder, cada bloque contiene:
# Multi-Head Self-Attention, MLP, Residual connections, LayerNorm (Es la "profundidad" del modelo)
param['n_blocks'] = 8

# Cada bloque usa 4 cabezas de atención. Atiende a distintas relaciones espaciales , captura diferentes tipos de patrones
param['n_heads'] = 4

# Cada parche (que originalmente tiene tamaño 32×32×1 = 1024 valores) se proyecta a un vector de dimensión 64. R^1024 ->R^64
param['projection_dim'] = 64

# Define el MLP interno de cada bloque transformer. 64→128→64, el parche tiene dimension 64 que se expande a 128 donde se expande el espacio de 
# representacion para combinar carct y crear interacciones no lineales, luego se vuelve a comprimir a 64 para manetener dimension usando  conexiones residuales.
param['transformer_units'] = [128, 64]

# Se trata del MLP final despues del transformer, es el que convierte el embedding global de la imagen en prediccion diagnostica, 64→512→256→clases
param['mlp_head_units'] = [512, 256] # [2048, 1024]

#---------------------------------------------------------------------------------------------------------------------------------------------
#El Transformer aprende una representación general.
#La MLP final aprende la frontera de decisión.

#### Training

In [6]:
# ENTRENAMIENTO DEL MODELO - COMO OPTIMIZAR ENTRENAMIENTO

# Tamaño del batch -> En cada paso de entrenamiento el modelo procesa 64 imagenes a la vez
# Si el input es (256,256,1) el tensor real que entra al modelo es (64,256,256,1)
param['batch_size'] = 64

# Elecccion entre CPU (Pocos nucleos(potentes), tareas secuenciales y complejas, latencia baja (rapida para 1 tarea))
# y GPU (Miles de nucleos (simples), tareas paralelas y masivas, latencia alta (rapida para muchas tareas))
# -1 if 'cpu', >=0 for 'gpu' selection
param['device'] = -1

# Path donde se guardara el modelo (Pesos en .h5 o modelo completo)
param['model_save_path'] = '/home/ander/Escritorio/Workshop2025-2026/results'

# Es el tamaño del paso (𝜂) en el descenso por gradiente (𝜃=𝜃−𝜂∇𝐿)
param['lr'] = 0.001

# Rgularizacion (𝜆). Ojo esto es regularizion L2 = (𝐿𝑡𝑜𝑡𝑎𝑙=𝐿𝑑𝑎𝑡𝑎+𝜆∣∣𝑤∣∣^2) pero nosotros usamas AdamW
param['weight_decay'] = 0.0001

# Numero de epocas, el modelo vera el dataset completo 200 veces.
param['n_epochs'] = 200

# Normalmente en una epech hay #muestras/batch_size, sin embargo, aqui forzamos a que haya 500 batches por epoca y como cada batch tiene 64 imagenes,
# entonces 500x64 = 3200 imagenes por epoca. (Se debe a que estaremos usando un generador con augmentation infinita)
param['steps_per_epoch'] = 500

# En cada epoch se evaluan 300×64=19200 imagenes de validacion
param['validation_steps'] = 300

In [7]:
print('Introduced all training parameters!')

print('Selected input parameters (training part): \n')

for param_name, param_value in param.items():
  print('- ' + param_name + ': ' + str(param_value))

Introduced all training parameters!
Selected input parameters (training part): 

- training_database_path: ['/home/ander/Escritorio/Workshop2025-2026/Databases/Mice_ecos/Ecos_tutorials_AI/ecos_orig/ecos_orig_Training']
- validation_database_path: ['/home/ander/Escritorio/Workshop2025-2026/Databases/Mice_ecos/Ecos_tutorials_AI/ecos_orig/ecos_orig_Validation']
- target_size: (256, 256)
- num_channels: 1
- patch_size: 32
- n_blocks: 8
- n_heads: 4
- projection_dim: 64
- transformer_units: [128, 64]
- mlp_head_units: [512, 256]
- batch_size: 64
- device: -1
- model_save_path: /home/ander/Escritorio/Workshop2025-2026/results
- lr: 0.001
- weight_decay: 0.0001
- n_epochs: 200
- steps_per_epoch: 500
- validation_steps: 300


### Device selection/check

In [8]:
# region Device Initialization/check

device_name_tf = tf.test.gpu_device_name()
print(device_name_tf)


os.environ["CUDA_VISIBLE_DEVICES"]=str(param['device'])
if param['device'] < 0:
    print('CPU selected')
else:
    print('GPU '  + str(param['device']) + ' selected')


# endregion


CPU selected


### Load data

#### Normalizer for training and validation

In [9]:
# Modulo de preprocesamiento y aumentacion de datos. ¡OJOOO : AQUI DEFINIMOS EL DATA AUGMENTATION, TODAVIA NO LO EJECUTAMOS!
data_augmentation = my_datasets.data_augmentation_pipeline(param['target_size'])

#### Get input data

In [10]:
# Cargamos dataset de entrenamiento
x_train, y_train, x_train_filenames, y_train_class_names, train_class_names, train_label_encoder = my_datasets.load_dataset_organized_sequences_different_sources(
    param['training_database_path'], img_size=param['target_size'],
    label_encoder=None)

# Cargamos dataset de validacion
x_val, y_val, x_val_filenames, y_val_class_names, val_class_names, val_label_encoder = my_datasets.load_dataset_organized_sequences_different_sources(
    param['validation_database_path'], img_size=param['target_size'],
    label_encoder=train_label_encoder) # Atento que aqui recicla el label encoder

# Adaptamos la capa de normalizacion (SINO NO SE PODRIA EJECUTAR LA NORMALIZACION, CALCULA μ y σ SOLO CON LOS DATOS DE ENTRENAMIENTO)
data_augmentation.layers[0].adapt(x_train)

Situation of samples with known classes
Database with 15554 samples and 4 classes
Loading data...
Numpy format
Number of loaded images: 0
Number of loaded images: 20
Number of loaded images: 40
Number of loaded images: 60
Number of loaded images: 80
Number of loaded images: 100
Number of loaded images: 120
Number of loaded images: 140
Number of loaded images: 160
Number of loaded images: 180
Number of loaded images: 200
Number of loaded images: 220
Number of loaded images: 240
Number of loaded images: 260
Number of loaded images: 280
Number of loaded images: 300
Number of loaded images: 320
Number of loaded images: 340
Number of loaded images: 360
Number of loaded images: 380
Number of loaded images: 400
Number of loaded images: 420
Number of loaded images: 440
Number of loaded images: 460
Number of loaded images: 480
Number of loaded images: 500
Number of loaded images: 520
Number of loaded images: 540
Number of loaded images: 560
Number of loaded images: 580
Number of loaded images: 

2026-02-19 10:39:29.990994: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 4077387776 exceeds 10% of free system memory.


### Batch generator

In [11]:
# Pasamos de arrays completos en memoria a batches.
train_dataset = batch_generator.data_generator(x_train, y_train, train_class_names, param)
validation_dataset = batch_generator.data_generator(x_val, y_val, val_class_names, param)

### Model design

In [12]:
# Construye la arquitectura ViT
# Integra la data augmentation
# Configura todas las dimensiones
# Devuelve un modelo Keras listo para entrenar

model_train = model.create_vit_classifier(data_augmentation,
                                          input_shape = (param['target_size'][0],param['target_size'][1], param['num_channels']),
                                          image_size = max(param['target_size'][0], param['target_size'][1]),
                                          patch_size=param['patch_size'],
                                          transformer_layers = param['n_blocks'],
                                          num_heads = param['n_heads'],
                                          projection_dim = param['projection_dim'],
                                          transformer_units = param['transformer_units'],
                                          mlp_head_units = param['mlp_head_units'],
                                          num_classes = len(train_class_names),
                                          vanilla=False)

model_train.summary() #Imprime la arquitectura completa con las capas, formas de entrada y salida, #params

print('Selected ViT model for small datasets in Keras')

Model: "data_augmentation"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 normalization (Normalizatio  (None, 256, 256, 1)      3         
 n)                                                              
                                                                 
 resizing (Resizing)         (None, 256, 256, 1)       0         
                                                                 
 random_flip (RandomFlip)    (None, 256, 256, 1)       0         
                                                                 
 random_rotation (RandomRota  (None, 256, 256, 1)      0         
 tion)                                                           
                                                                 
 random_zoom (RandomZoom)    (None, 256, 256, 1)       0         
                                                                 
Total params: 3
Trainable params: 0
Non-trainable

### Compiling and optimizer

In [13]:
# steps por epoch x epochs
total_steps = int((len(x_train_filenames) / param['batch_size']) * param['n_epochs'])
# Usamos Adam + Weigth decay 
optimizer = tfa.optimizers.AdamW(learning_rate=param['lr'], weight_decay=param['weight_decay'])

# Definimos 3 cosas:
# 1. la primera es el optimizador (como se actualizan los pesos)
# 2. Luego, definimos la funcion loss (como medir el error), que se debe a que la codificacion no es one-hot sino enteros 
# y from_logits=True indica que la ultima capa del modelo NO tiene softmax
# 3. (Como medir el rendimiento), total/predicciones correctas
model_train.compile(optimizer=optimizer,
                    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")])

### Callbacks

Called during fitting process.

### Fitting process

This code will take hours to run on a CPU. We recommend you to skip this step here and load the proposed pretrained weights so as to see how testing works.

In [14]:
model_history = model_train.fit(train_dataset, validation_data = validation_dataset,
                                   batch_size = param['batch_size'],
                                   epochs = param['n_epochs'],
                                   steps_per_epoch = param['steps_per_epoch'],
                                   validation_steps = param['validation_steps'],
                                   callbacks = callbacks.callbacks(param, model_train, train_class_names, validation_data = [x_val, y_val], total_steps = total_steps)
                                   )

Model Checkpoint
Done!


Batch size: 64
Number of minimum samples per class in each batch: 16
Number of remaining samples per class in each batch: 0

Class 0: 6046
Class 1: 3141
Class 2: 3367
Class 3: 3000
Epoch 1/200


2026-02-19 10:40:29.799481: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype int32
	 [[{{node Placeholder/_0}}]]
2026-02-19 10:40:36.772178: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 83886080 exceeds 10% of free system memory.
2026-02-19 10:40:36.786699: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 83886080 exceeds 10% of free system memory.
2026-02-19 10:40:36.808946: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 83886080 exceeds 10% of free system memory.
2026-02-19 10:40:36.827848: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 83886080 exceeds 10% of free system memory.


500/500 [==============================] - ETA: 0s - loss: 1.8360 - accuracy: 0.2473

Batch size: 64
Number of minimum samples per class in each batch: 16
Number of remaining samples per class in each batch: 0

Class 0: 2936
Class 1: 2100
Class 2: 3000
Class 3: 3068


2026-02-19 10:47:09.432668: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype int32
	 [[{{node Placeholder/_0}}]]



Epoch 1: val_accuracy improved from -inf to 0.25000, saving model to /home/ander/Escritorio/Workshop2025-2026/results__2026-02-19_10-40/saved_model_epoch_001.h5
{'loss': 1.8359829187393188, 'accuracy': 0.2473125010728836, 'val_loss': 1.386777400970459, 'val_accuracy': 0.25}
---------------------GPU USAGE---------------------
nvidia-smi not found. Make sure NVIDIA GPU drivers and CUDA are installed.
---------------------END GPU USAGE---------------------
---------------------RAM USAGE---------------------
Total Memory: 15.5 GB
Available Memory: 6.0 GB
Used Memory: 9.5 GB
Memory Usage Percentage: 61.4 %
---------------------END RAM USAGE---------------------
500/500 [==============================] - 533s 1s/step - loss: 1.8360 - accuracy: 0.2473 - val_loss: 1.3868 - val_accuracy: 0.2500 - lr: 1.0267e-04
Epoch 2/200
500/500 [==============================] - ETA: 0s - loss: 1.3943 - accuracy: 0.2511
Epoch 2: val_accuracy did not improve from 0.25000
{'loss': 1.3943029642105103, 'accurac

KeyboardInterrupt: 